# 03｜分類模型與評估

**高一學生實作版｜讀取 Iris CSV → LogisticRegression → 混淆矩陣 → DecisionTree**

本週只使用一份資料：**`iris_classification.csv`**。其中有 **150 朵鳶尾花、3 種類別**；我們用花瓣長度與花瓣寬度訓練模型，預測是哪一種花。

### 課前準備（請先完成）

1. 將 `iris_classification.csv` 下載到電腦，**在 Colab 開始第一題前先上傳檔案**。VS Code Notebook 可把 CSV 放在 Notebook 相同資料夾。
2. 每題先閱讀 Markdown 的 Python 範例，在**緊接的空白 Code cell** 自己輸入，按 **Shift + Enter** 執行，再完成「輪到你」。
3. **依順序執行：** 後面的練習使用先前建立的 `df`、`X`、`y`、`model` 等變數；若重新啟動執行環境，請從第一題重新輸入執行。
4. 所需套件：`pandas`、`numpy`、`matplotlib`、`scikit-learn`。Colab 通常已預先安裝。

> **重點：** CSV 的 `species` 已寫有 `Setosa`、`Versicolor`、`Virginica`，可直接當作 y（分類標籤），不用額外下載資料集或把種類轉成 0／1／2。

---
## 第一節｜分類概念與 Iris 資料集

先理解 **樣本（sample）**、**特徵（feature, X）** 和 **標籤（label, y）**，接著畫散佈圖，最後分出訓練集和測試集。

**分類與回歸不同：** 預測「花瓣有多長」是回歸；預測「是哪一種花」是分類。

### 練習 1-1｜讀取 Iris CSV

`pd.read_csv()` 讀取教師提供的 CSV。`shape` 可看到資料的（列數, 欄數）。**如果出現 FileNotFoundError，請先把 `iris_classification.csv` 上傳到 Colab 的檔案區。**

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
import pandas as pd

df = pd.read_csv("iris_classification.csv")
print("資料形狀：", df.shape)
print(df.head())
```

**輪到你：** 把 `head()` 改成 `head(3)`，觀察前三筆資料。

### 練習 1-2｜認識四個測量欄位與種類

資料有 `sepal_length`、`sepal_width`、`petal_length`、`petal_width` 四種測量值（單位：公分），以及 `species` 花朵種類；`id` 只是資料編號。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
print(df.columns.tolist())
print(df.loc[0, ["petal_length", "petal_width", "species"]])
```

**輪到你：** 將 `loc[0]` 改成 `loc[50]`，比較不同種類的花瓣數值。

### 練習 1-3｜建立 X（特徵）與 y（標籤）

一列資料是一朵花。選擇花瓣長與花瓣寬當作 **X**；CSV 中的 `species` 文字就是 **y**。`.to_numpy()` 讓後續的模型取得陣列。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
X = df[["petal_length", "petal_width"]].to_numpy()
y = df["species"].to_numpy()
print("X：", X.shape)
print("y：", y.shape)
print("第一筆：", X[0], y[0])
```

**輪到你：** 印出 `X[50]` 和 `y[50]`，理解測量值與真正類別的關係。

### 練習 1-4｜使用 Pandas 查看三種類別

不需要重新建立 CSV。直接用 `value_counts()` 計算各類有多少筆。這份資料共有三類，各 50 筆。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
print(df["species"].value_counts())
print(df[["petal_length", "petal_width"]].describe())
```

**輪到你：** 觀察花瓣長度與寬度的最小值、最大值，想想不同花朵是否會重疊。

### 練習 1-5｜按照類別查看資料

可以透過 Pandas 條件篩選，找出某一種類的花朵；這與第一週的資料篩選概念相同。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
setosa = df[df["species"] == "Setosa"]
print("Setosa 筆數：", len(setosa))
print(setosa[["petal_length", "petal_width"]].head())
```

**輪到你：** 把 `"Setosa"` 換成 `"Virginica"`，看看花瓣尺寸。

### 練習 1-6｜畫三種鳶尾花的散佈圖

每個點代表一朵花，橫軸是花瓣長度、縱軸是花瓣寬度。使用 `df["species"] == name` 依類別選點。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
import matplotlib.pyplot as plt

names = ["Setosa", "Versicolor", "Virginica"]
for name in names:
    part = df[df["species"] == name]
    plt.scatter(part["petal_length"], part["petal_width"], label=name)
plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.legend()
plt.show()
```

**輪到你：** 指出哪兩類有重疊，以及哪一類較容易和其他類分開。

### 練習 1-7｜分類機率的概念：選最大的一類

這組機率是**教學假設**，還不是真正的模型預測。`argmax()` 找到最大值的位置，再用種類名稱對照。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
import numpy as np

prob = [0.1, 0.8, 0.1]
index = np.argmax(prob)
print("最大機率：", prob[index])
print("預測種類：", names[index])
```

**輪到你：** 把機率改成 `[0.8, 0.1, 0.1]`，看看類別如何改變。

### 練習 1-8｜切分訓練集與測試集

與投影片一致，150 筆分成訓練 **105 筆**、測試 **45 筆**。`stratify=y` 確保三種類別比例相近；`random_state=4` 固定切分。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=4, stratify=y
)
print("訓練：", len(X_train))
print("測試：", len(X_test))
print(pd.Series(y_test).value_counts())
```

**輪到你：** 比較訓練集三種類別的筆數，確認每類訓練 35 筆。

---
## 第二節｜LogisticRegression 與分類評估

依序完成：**建立模型 → fit → predict → predict_proba → Accuracy → 混淆矩陣 → 決策區域**。

這裡的 LogisticRegression 名稱雖然有 Regression，但它是**分類模型**；我們要預測的是花的類別，不是花瓣長度。

### 練習 2-1｜訓練 LogisticRegression 分類模型

`LogisticRegression()` 建立分類器，`fit(X_train, y_train)` 使用訓練資料學習如何分辨三種花。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("模型訓練完成")
```

**輪到你：** 將 `print` 的文字換成自己能理解的描述，說出 `fit()` 使用哪兩份資料。

### 練習 2-2｜predict：預測測試集種類

`predict(X_test)` 只需要測試花朵的**特徵 X**，就可以產生預測種類名稱。`y_test` 先保留，之後用來核對。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
pred = model.predict(X_test)
print("前五筆預測：", pred[:5])
print("前五筆真實：", y_test[:5])
```

**輪到你：** 將 `[:5]` 改成 `[:10]`，看看前十筆的預測是否全部相同。

### 練習 2-3｜比較真實類別與預測類別

因為 CSV 的 `species` 原本就有種類名稱，模型直接輸出 `Setosa`、`Versicolor` 或 `Virginica`，**不需要再轉換編號**。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
print("真實：", y_test[:5])
print("預測：", pred[:5])
print("是否相同：", y_test[:5] == pred[:5])
```

**輪到你：** 把前五筆改成前八筆，比較分類結果。

### 練習 2-4｜predict_proba：三種類別的機率

`model.classes_` 告訴我們機率的欄位順序。每筆資料的三類機率相加約為 1，**不代表預測一定正確**。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
prob = model.predict_proba(X_test[:1])
print("種類順序：", model.classes_)
print("各類機率：", prob[0])
print("加總：", prob[0].sum())
print("預測：", model.predict(X_test[:1])[0])
```

**輪到你：** 用 `X_test[1:2]` 比較另一朵花的機率。

### 練習 2-5｜自己輸入一朵新花來預測

假設有一朵新花，花瓣長 4.8 公分、寬 1.5 公分。特徵順序要與 `X` 相同：**先花瓣長，再花瓣寬**。模型的結果只是推測，不是真實標籤。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
new = [[4.8, 1.5]]
result = model.predict(new)
print("預測類別：", result[0])
```

**輪到你：** 把 `new` 改成 `[[1.4, 0.2]]`，看看比較小的花瓣會被判成哪一類。

### 練習 2-6｜Accuracy：測試集正確分類比例

`accuracy_score()` 計算**答對筆數 ÷ 測試筆數**。使用投影片相同的切分，這裡預計有 **44 筆答對、45 筆測試資料**，約為 97.8%。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, pred)
print("答對筆數：", sum(y_test == pred))
print("測試筆數：", len(y_test))
print("Accuracy：", round(acc, 3))
```

**輪到你：** 算一算 `44 ÷ 45` 大約是多少，再說明它是否代表對所有新花都會一樣準。

### 練習 2-7｜混淆矩陣：看出錯在哪一類

混淆矩陣的**列是真實類別**，**欄是預測類別**。對角線表示預測正確，其他格子表示誤判。使用 `ConfusionMatrixDisplay` 可直接把數字畫成圖。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, pred)
print(cm)
ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot()
plt.show()
```

**輪到你：** 找出矩陣中唯一的非對角線錯誤，說明「哪一類被判成哪一類」。

### 練習 2-8｜決策區域：不同顏色代表不同預測種類

分類器會對平面上的每個位置做預測。不同底色表示不同的**預測類別**，散佈點的顏色則代表花朵的**真實類別**。`pd.Categorical(y).codes` 只用於畫圖著色，**並未修改模型的 y 標籤**。靠近交界的樣本可能較難判斷。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.inspection import DecisionBoundaryDisplay

DecisionBoundaryDisplay.from_estimator(model, X, alpha=0.25)
plt.scatter(X[:, 0], X[:, 1], c=pd.Categorical(y).codes, edgecolors="k", s=20)
plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.show()
```

**輪到你：** 觀察 Setosa 與其他兩類的交界位置；哪些花朵比較靠近分類邊界？

---
## 第三節｜DecisionTree 與模型比較

同一批訓練集和測試集，嘗試 **DecisionTreeClassifier**。比較不同深度、查看訓練與測試的 Accuracy，最後認識 **Precision** 和 **Recall**。

### 練習 3-1｜建立一棵深度 2 的決策樹

決策樹像在問一連串「是／否」問題，沿著條件走到葉節點，就能給出花朵種類。`max_depth=2` 表示最多往下切兩層。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X_train, y_train)
print("決策樹測試 Accuracy：", tree.score(X_test, y_test))
```

**輪到你：** 把 `max_depth` 改成 `1`，重新執行後比較測試準確率。

### 練習 3-2｜把決策樹的條件畫出來

`plot_tree()` 可以將分支條件與葉節點畫出來。每一個方框都是一個判斷步驟或結果；讀圖時由上往下看。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.tree import plot_tree

plt.figure(figsize=(10, 5))
plot_tree(tree, feature_names=["Petal length", "Petal width"],
          class_names=list(tree.classes_), filled=True, rounded=True)
plt.show()
```

**輪到你：** 從根節點往下走一次，找出花瓣寬度很小的花通常會被分到哪一類。

### 練習 3-3｜同一朵新花，兩種模型會怎麼預測？

沿用第二節的 `model`（邏輯斯迴歸）以及本節的 `tree`（決策樹），把**完全相同的花瓣資料**交給兩種模型。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
new = [[4.8, 1.5]]
print("邏輯斯迴歸：", model.predict(new)[0])
print("決策樹：", tree.predict(new)[0])
```

**輪到你：** 把花瓣寬改成 `2.0` 公分，再比較兩個模型的預測。

### 練習 3-4｜改變 max_depth，記錄兩組 Accuracy

樹越深，允許的判斷越多；**訓練 Accuracy 變好，不代表測試 Accuracy 一定更好**。先記錄深度 1～5 的分數，下一題再畫圖。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
depths = [1, 2, 3, 4, 5]
train_acc = []
test_acc = []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_train, y_train)
    train_acc.append(tree.score(X_train, y_train))
    test_acc.append(tree.score(X_test, y_test))
print("訓練：", train_acc)
print("測試：", test_acc)
```

**輪到你：** 比較深度 1 與深度 5 的兩組 Accuracy。測試分數有隨深度一直提高嗎？

### 練習 3-5｜畫出訓練與測試 Accuracy 曲線

將上一題的深度與準確率畫成折線圖。若訓練分數上升而測試分數沒有一起改善，要留意模型是否只記住訓練資料的細節。**單次切分還不能直接證明過度擬合。**

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
plt.plot(depths, train_acc, marker="o", label="Train")
plt.plot(depths, test_acc, marker="o", label="Test")
plt.xlabel("Tree depth")
plt.ylabel("Accuracy")
plt.legend()
plt.show()
```

**輪到你：** 指出測試 Accuracy 最高的深度；如果有不只一個深度並列，哪個模型較簡單？

### 練習 3-6｜公平比較兩種模型的預測結果

回到深度 3 的決策樹，與 LogisticRegression 使用**同一批** `X_test`、`y_test` 比較，才能公平判讀。除了 Accuracy，也檢查混淆矩陣。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)
print("邏輯斯迴歸：", round(accuracy_score(y_test, pred), 3))
print("決策樹：", round(accuracy_score(y_test, tree_pred), 3))
print(confusion_matrix(y_test, tree_pred))
```

**輪到你：** 比較兩個模型的 Accuracy；就算分數一樣，錯誤的種類也可能不一樣。

### 練習 3-7｜Precision 與 Recall：哪種錯誤比較重要？

指定某一類是「正類」後，**Precision（精確率）**看「被預測為這一類」的有多少真的屬於它；**Recall（召回率）**看「真正是這一類」的有多少被找回。以下對三種花各算一次。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
from sklearn.metrics import precision_score, recall_score

p = precision_score(y_test, tree_pred, average=None, zero_division=0)
r = recall_score(y_test, tree_pred, average=None, zero_division=0)
for i in range(3):
    print(tree.classes_[i], "Precision:", round(p[i], 3),
          "Recall:", round(r[i], 3))
```

**輪到你：** 比較 Versicolor 和 Virginica 的 Recall。若最擔心漏掉某一類，應注意哪個數字？

### 練習 3-8｜完整小任務：預測新花，並解釋評估結果

整理本週流程：**選兩個特徵 → 切分 → fit → predict → 看 Accuracy 和混淆矩陣**。結論要包含樣本數、特徵、模型、測試結果，也要提醒預測不是百分之百保證。

**參考程式（先閱讀，再輸入下方的空白 Code cell）：**

```python
new = [[4.8, 1.5]]
result = tree.predict(new)
acc = accuracy_score(y_test, tree_pred)
cm = confusion_matrix(y_test, tree_pred)
print("新花預測：", result[0])
print("測試 Accuracy：", round(acc, 3))
print("混淆矩陣：")
print(cm)
```

**輪到你：** 修改新花瓣的長、寬各一次，並用兩句話說明你的模型「預測哪一類」與「有哪些限制」。

---
## 完成後，你應該能夠……

- 分辨**分類**與**回歸**，並指出樣本、特徵 `X`、標籤 `y`。
- 使用 Pandas `read_csv()` 讀入 `iris_classification.csv`，辨識四種測量值與 `species` 標籤。
- 畫出兩個花瓣特徵的散佈圖，了解分類邊界。
- 使用 `train_test_split(..., stratify=y)` 切出三類比例相近的訓練集、測試集。
- 使用 `LogisticRegression` 和 `DecisionTreeClassifier` 完成 `fit()` 與 `predict()`。
- 使用 **Accuracy**、**混淆矩陣**、**Precision**、**Recall** 說明模型的表現和限制。

**交作業前檢查：** 24 個空白 Code cell 是否都已自己輸入、執行？圖表是否看得到？有沒有將測試集拿去重新 `fit()`？

> **提醒：** Iris 是練習資料集。這次高分只代表固定切分下的測試結果，不保證模型對所有新資料都一樣準確。